In [ ]:
# =====================================================================
# dataset_stats.py — how much LABELED data per species (rerun anytime).
# =====================================================================
# Counts, per species, how many images now have a YOLO box (auto + hand),
# vs how many were downloaded. Pure analysis — touches nothing. Run it as
# often as you like (unlike second_pass.py which does real detection work).
#
#   python dataset_stats.py
#
# Reads: dataset/manifest.json, dataset/labels/{train,val}/*.txt
# Prints + writes dataset/data_stats.txt and dataset/data_stats.csv

import json, csv
from pathlib import Path
from collections import defaultdict

OUT = Path("dataset")
manifest = json.loads((OUT / "manifest.json").read_text())

# which stems currently have a (non-empty) label file?
labeled = set()
boxes_per_stem = {}
for split in ["train", "val"]:
    for lp in (OUT / "labels" / split).glob("*.txt"):
        txt = lp.read_text().strip()
        if txt:
            labeled.add(lp.stem)
            boxes_per_stem[lp.stem] = len(txt.splitlines())

per_species = defaultdict(lambda: {"downloaded": 0, "labeled": 0, "boxes": 0,
                                    "coarse": "?"})
for m in manifest:
    s = per_species[m["species"]]
    s["downloaded"] += 1
    s["coarse"] = m["coarse"]
    if m["stem"] in labeled:
        s["labeled"] += 1
        s["boxes"] += boxes_per_stem.get(m["stem"], 0)

rows = []
for sp, d in per_species.items():
    cov = 100 * d["labeled"] / max(d["downloaded"], 1)
    rows.append((sp, d["coarse"], d["downloaded"], d["labeled"], d["boxes"], cov))

# sort worst coverage first — these need the most hand-labeling
rows.sort(key=lambda r: r[5])

# CSV
with open(OUT / "data_stats.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["species", "class", "downloaded", "labeled", "boxes", "coverage_%"])
    for r in rows:
        w.writerow([r[0], r[1], r[2], r[3], r[4], f"{r[5]:.0f}"])

# text report
lines = ["================ DATA PER SPECIES (worst coverage first) ================",
         f"{'cov':>4} {'lbl':>5}/{'dl':<5} {'boxes':>6}  species"]
for sp, coarse, dl, lbl, bx, cov in rows:
    lines.append(f"{cov:3.0f}% {lbl:>5}/{dl:<5} {bx:>6}  [{coarse[0]}] {sp}")

tot_dl = sum(r[2] for r in rows); tot_lbl = sum(r[3] for r in rows)
tot_bx = sum(r[4] for r in rows)
lines += ["",
          f"TOTAL: {tot_lbl}/{tot_dl} images labeled "
          f"({100*tot_lbl//max(tot_dl,1)}%), {tot_bx} boxes across "
          f"{len(rows)} species.",
          "Species under ~50% coverage are your hand-labeling priority "
          "(the label tool already orders them first)."]

report = "\n".join(lines)
(OUT / "data_stats.txt").write_text(report)
print(report)
print(f"\nSaved dataset/data_stats.txt and dataset/data_stats.csv")

================ DATA PER SPECIES (worst coverage first) ================
 cov   lbl/dl     boxes  species
 15%    54/350       67  [m] Siberian Flying Squirrel
 29%    13/45        14  [m] American beaver
 30%    20/67        23  [b] Great Bittern
 35%    26/74        35  [b] Water Rail
 43%    28/65        34  [m] gray wolf
 43%    25/58        31  [m] muskrat
 46%    31/68        32  [b] Ortolan Bunting
 49%    34/69        40  [b] Eurasian Nightjar
 49%   173/350      228  [m] European otter
 50%    86/172      101  [b] Tawny Owl
 50%    43/86        44  [b] Blyth's Reed Warbler
 51%    38/74        46  [b] Lapland Longspur
 53%    81/153       84  [m] Asian Minute Shrew
 53%    42/79        53  [m] Northern Bat
 55%    70/127      103  [b] Parrot Crossbill
 57%    28/49        37  [b] Carrion Crow
 59%    50/85        52  [b] Garden Warbler
 59%    92/156       98  [b] Rustic Bunting
 60%    46/77        88  [b] Common Scoter
 60%    58/97        73  [b] Red-breasted Flycatcher
 6

In [ ]:
# =====================================================================
# eval_per_species.py — how well does the trained detector do, PER SPECIES?
# =====================================================================
# The detector only knows bird/mammal (2 classes), so standard mAP won't
# tell you "how well does it find a redwing vs a wren". This evaluates per
# SPECIES by running the model on each species' val images and measuring
# whether it found the animal (recall) and how confident/tight the box was.
#
# It uses the val split + manifest to group images by species, runs the
# trained detector, and reports per-species detection rate + mean IoU
# against the ground-truth boxes.
#
#   python eval_per_species.py --weights runs/birdpoke/s/weights/best.pt
#
# Reads: dataset/manifest.json, dataset/images/val, dataset/labels/val
# Writes: dataset/eval_per_species.txt and .csv

import argparse, json, csv
from pathlib import Path
from collections import defaultdict

from ultralytics import YOLO

OUT = Path("dataset")
DETECTOR_CLASSES = ["bird", "mammal"]
IOU_HIT = 0.5            # a detection counts as correct if IoU>=this with a GT box


def load_gt(stem, split):
    lp = OUT / "labels" / split / f"{stem}.txt"
    if not lp.exists():
        return []
    boxes = []
    for line in lp.read_text().splitlines():
        if not line.strip():
            continue
        c, cx, cy, w, h = map(float, line.split())
        boxes.append((int(c), cx - w/2, cy - h/2, cx + w/2, cy + h/2))  # xyxy norm
    return boxes


def iou(a, b):
    x1 = max(a[0], b[0]); y1 = max(a[1], b[1])
    x2 = min(a[2], b[2]); y2 = min(a[3], b[3])
    inter = max(0, x2-x1) * max(0, y2-y1)
    ua = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter/ua if ua > 0 else 0


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--weights", default="runs/birdpoke/s/weights/best.pt")
    ap.add_argument("--conf", type=float, default=0.25)
    args = ap.parse_args()

    manifest = json.loads((OUT / "manifest.json").read_text())
    val_by_species = defaultdict(list)
    for m in manifest:
        if m["split"] == "val":
            val_by_species[m["species"]].append(m)

    model = YOLO(args.weights)

    rows = []
    for species, items in val_by_species.items():
        coarse = items[0]["coarse"]
        n = 0; detected = 0; iou_sum = 0.0; iou_n = 0
        for m in items:
            p = OUT / "images" / "val" / f"{m['stem']}.jpg"
            if not p.exists():
                continue
            gt = load_gt(m["stem"], "val")
            if not gt:
                continue
            n += 1
            res = model.predict(str(p), conf=args.conf, imgsz=640, verbose=False)[0]
            W, H = res.orig_shape[1], res.orig_shape[0]
            preds = []
            for b in res.boxes:
                x1, y1, x2, y2 = [float(v) for v in b.xyxy[0]]
                preds.append((x1/W, y1/H, x2/W, y2/H))
            # did any prediction hit any GT box?
            best = 0.0
            for g in gt:
                for pr in preds:
                    best = max(best, iou(g[1:], pr))
            if best >= IOU_HIT:
                detected += 1
            if preds:
                iou_sum += best; iou_n += 1
        if n == 0:
            continue
        rec = 100*detected/n
        mean_iou = iou_sum/max(iou_n, 1)
        rows.append((species, coarse, n, detected, rec, mean_iou))

    rows.sort(key=lambda r: r[4])   # worst recall first

    with open(OUT / "eval_per_species.csv", "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["species","class","val_imgs","detected","recall_%","mean_iou"])
        for r in rows:
            w.writerow([r[0], r[1], r[2], r[3], f"{r[4]:.0f}", f"{r[5]:.2f}"])

    lines = [f"========= PER-SPECIES DETECTION ({args.weights}) =========",
             f"{'recall':>6} {'iou':>5} {'det':>4}/{'val':<4}  species"]
    for sp, coarse, n, det, rec, miou in rows:
        lines.append(f"{rec:5.0f}% {miou:5.2f} {det:>4}/{n:<4}  [{coarse[0]}] {sp}")
    overall = sum(r[3] for r in rows)/max(sum(r[2] for r in rows), 1)*100
    lines += ["", f"Overall detection recall: {overall:.1f}% across {len(rows)} species.",
              "Low-recall species: too few/poor training boxes, or visually hard.",
              "Cross-reference with data_stats.py — usually low recall == low data."]
    report = "\n".join(lines)
    (OUT / "eval_per_species.txt").write_text(report)
    print(report)


if __name__ == "__main__":
    main()